In [2]:
import pandas as pd
import pickle
import python_files.paths_mapping as paths
import python_files.functions_barcelona as fb
import utm
import folium
import numpy as np
import geopandas as gpd
import os
import webbrowser
%reload_ext autoreload

/Users/fcbnyc/mystuff/repos/BarcelonaAccidents2024/Tableau/shapefiles


In [42]:
def fixing_columns(dataframe):
    """
        Changes column names to have the same column names
    """
    dataframe.columns =[col.replace("ú",'u').replace("ó","o").replace("'",'').replace(' ','_').replace("_de_","_").replace("í","i").replace("¢",'o') for col in dataframe]
    dataframe = dataframe.rename(columns=paths.mapping_original_columns,)
   
    return dataframe
    



def fixing_lat(row):
    if row['lat'] == -1:
        return row['lon_lat_check'] + 1000
    if pd.isnull(row['lat']) is None:
        return row['lon_lat_check'] + 10000
    if 41.5>row['lat'] >41.2000:
        return row['lon_lat_check']
    else:
        return row['lon_lat_check'] + 1
        
def fixing_lon(row):
    if row['lon'] == -1:
        return row['lon_lat_check'] + 1000
    if pd.isnull(row['lon']) is None:
        return row['lon_lat_check'] + 10000
    if 2.27>row['lon'] >2.06:
        return row['lon_lat_check']
    else:
        return row['lon_lat_check'] + 1

def read_df(base_dir,pathname):
    print(f'Working with...{pathname}')
    encodings = ['utf-8', 'latin-1']
    for enc in encodings:
        try:
            df = pd.read_csv(base_dir+pathname, encoding = enc, engine='python')
            print(f"{pathname} successfully read with {enc}")
            return df
            break
        except:
            try:
                df = pd.read_csv(base_dir+pathname, encoding = enc, sep=';')
                print(f"{pathname} successfully read with {enc}")
                return df
                break
            except UnicodeDecodeError:
                continue
    else:
        raise Exception(f"unable to decode the file {pathname}")

In [143]:

columnes=['num_incident', 'district_code','district', 'neighborhood','street_name',
          'weekday', 'year', 'month', 'day', 'hour', 'ped_cause',
           'num_deaths', 'num_minorly_injured', 'num_severly_injured',
            'num_victims', 'num_vehicles','lon', 'lat' ]
    
##ACCIDENTS
year =2025
prename ='accidents-gu'
print(f'################{prename}##############')

#1. Recovering previous years
old_df =pd.read_csv(paths.FILES_PATH/f'accidents_only{year-1}.csv')
#2. Loading current year
with open(paths.FILES_PATH /f'dict_files_{year}.pkl','rb') as file:
    dicty = pickle.load(file)
df = dicty[prename]
df = fixing_columns(df)
df.columns = [col.lower() for col in df.columns]
#fixing coor
df['lon_lat_check'] =[0]*len(df)
df[['lat','lon']] =df[['lat','lon']].astype(float)

df['lon_lat_check'] = df.apply(fixing_lat,axis=1)
df['lon_lat_check'] = df.apply(fixing_lon,axis=1)
print(year,df.lon_lat_check.value_counts())

to_check = df[df.lon_lat_check !=0].index

print(f"SOMETHING TO DELETE in {year}?:",len(to_check))

to_check = df[df.lon_lat_check !=0].index
print('Rows deleted: ', len(to_check))
df = df.drop(to_check)
print("Rows to be fixed: ", df[df.lon_lat_check !=0].shape[0])
#Assigning new columns
df = df[columnes]
print(df.shape)

#fixing/translating columns
df['num_incident'] = df['num_incident'].str.strip()
#print(df.ped_cause.value_counts())
df['ped_cause']=df.apply(fb.ped_to_angles, axis=1)
cols_with_nulls = list(df.isnull().sum()[df.isnull().sum()>0].index)
cols_with_nulls
for col in cols_with_nulls:
    df[col] =df[col].fillna(0).astype(int)
#df['num_victims'] = df['num_victims'].astype(int)
df['num_victims'] =df['num_deaths']+df['num_minorly_injured']+df['num_severly_injured']
df['num_victims'] = df['num_victims'].astype(int)
assert df.isnull().sum().sum() == 0, "FIX NULLS"

df['weekday']=df.apply(fb.setmana_a_angles,axis =1)
# print(df.weekday.value_counts())
df['month']=df.apply(fb.mes_a_angles, axis=1)
#print(df.month.value_counts())
#fix dups
#create datetime
print(df.shape)
df=df.drop_duplicates(subset='num_incident',keep='last')
print(df.shape)

final_df = pd.concat([old_df, df])

print('FINAL SHAPE: ', final_df.shape)
assert final_df.shape[1] == 18, f"Some new columns are appearing in {year}"
#fixing nulls
assert final_df.isnull().sum().sum() == 0, "FIX NULLS" 
final_df.to_csv(paths.FILES_PATH/f'accidents_only{year}.csv',index=False)
#os.remove(paths.FILES_PATH/f'accidents_only{year-1}.csv')
print(f"DONE {prename}")

###########################

prename ='causes'
print(f'################{prename}##############')
old_df =pd.read_csv(paths.FILES_PATH/f'causes{year-1}.csv')
old_df.columns
df = dicty[prename]
df =fixing_columns(df)
columnes =['num_incident','cause']
df = df[columnes]
df['cause']=df.cause.apply(fb.posant_accents).apply(fb.cause_to_angles)
#Check for nulls
assert df.isnull().sum().sum() == 0, "Must deal with NULLS"
#fix incident
df['num_incident'] = df['num_incident'].str.strip()
pre = df.shape[0]

df=df.drop_duplicates('num_incident',keep='last')
print('Rows dropped: ',pre - df.shape[0])

final_df = pd.concat([old_df,df])

assert len(old_df) + len(df) == len(final_df), "Something happened while concat"

assert final_df.isnull().sum().sum() == 0, "FIX NULLS" 
final_df.to_csv(paths.FILES_PATH/f'causes{year}.csv',index=False)
print(f"DONE {prename}")

########################
prename ='persones'
print(f'################{prename}##############')
old_df =pd.read_csv(paths.FILES_PATH/f'people{year-1}.csv')

df = dicty[prename]
df =fixing_columns(df)
columnes =['num_incident','cause']
df.rename(columns=paths.mapping_original_columns,inplace=True)
chosen_features=['num_incident','gender','age','people_role','vehicle','injuries_degree']
df = df[chosen_features]
df.rename(columns=paths.mapping_original_columns,inplace=True)

# #Taking care of duplicated column names
df['num_incident'] =df['num_incident'].str.strip()

df['gender'] = df['gender'].map(paths.mapping_gender)
df['people_role'] = df['people_role'].map(paths.mapping_role)
df['vehicle'] = df['vehicle'].map(paths.mapping_vehicles)

df['injuries_degree'] = df['injuries_degree'].map(paths.mapping_injuries)

print('Criteria to group vehicles:', paths.vehicle_grouping_criteria)
df['vehicle_group'] = df.apply(fb.assigning_vehicle_to_group,axis=1)
df['age'] = df.age.replace('Desconegut',np.NaN).astype(float)
df['age'] = df['age'].replace(-1,np.NaN)
df['gender'] = df['gender'].replace(-1,np.NaN)

print('NULLS: ', df.isnull().sum().sum())
if df.isnull().sum().sum() !=0:
    print(df.isnull().sum())
a = len(old_df)
b =len(df)
final_df = pd.concat([old_df,df])
final_df = fb.imputing_nulls(final_df)
assert len(final_df) == a+ b, "Check the CONCAT"
assert final_df.isnull().sum().sum() == 0,'Check NULLS'

final_df.to_csv(paths.FILES_PATH/f'people{year}.csv',index=False)

print('LENGTH: ', final_df.shape)
print(f"DONE {prename}")
########################
prename ='tipus'
print(f'################{prename}##############')
old_df =pd.read_csv(paths.FILES_PATH/f'types{year-1}.csv')
df = dicty[prename]
df =fixing_columns(df)
columnes =['num_incident','accident_type']
df.rename(columns=paths.mapping_original_columns,inplace=True)
df = df[columnes]
# #Taking care of duplicated column names
df['num_incident'] = df['num_incident'].str.strip()
df['accident_type']= df['accident_type'].replace('','Desconegut')
df['accident_type'] = df['accident_type'].map(paths.mapping_types)
df.accident_type.value_counts(dropna=False)
print(f'Criteria to group {prename}:', paths.type_grouping_criteria)
df['type_group'] = df.apply(fb.assigning_type_to_group,axis=1)
print("DUPLICATES: ",df[df.duplicated()].shape[0])
print('NULLS: ', df.isnull().sum().sum())
if df.isnull().sum().sum() !=0:
    print(df.isnull().sum())
a = len(old_df)
b =len(df)
final_df = pd.concat([old_df,df])
assert len(final_df) == a+ b, "Check the CONCAT"
assert final_df.isnull().sum().sum() == 0,'Check NULLS'

final_df.to_csv(paths.FILES_PATH/f'types{year}.csv',index=False)

print('LENGTH: ', final_df.shape)
print(f"DONE {prename}")
########################
prename ='vehicles'
print(f'################{prename}: by the far the less relevant one##############')
old_df =pd.read_csv(paths.FILES_PATH/f'vehicles{year-1}.csv')
df = dicty[prename]
df =fixing_columns(df)
columnes =['num_incident','vehicle_model','vehicle_brand_name',
           'vehicle_color','license','license_seniority']
df = df[columnes]

#incident to strip
df['num_incident'] = df['num_incident'].str.strip()
print("DUPLICATES: ",df[df.duplicated()].shape[0])
#vehicle brand name: lower and decide about desconegut
df['vehicle_brand_name'] = df['vehicle_brand_name'].str.lower().str.replace('desconegut','unknown')
#vehicle model: lower and decide about desconegut
df['vehicle_model'] = df['vehicle_model'].str.lower().replace('desconegut','unknown').replace('no disponible','unknown').replace('','unknown')
#vehicle color: map names and replace desconegut
df['vehicle_color'] = df['vehicle_color'].str.lower().replace('','unknown').replace('desconegut','unknown')

df['vehicle_color'] = df['vehicle_color'].map(paths.mapping_colors)
#vehicle license
df['license'] = df['license'].str.lower().replace('desconegut','unknown').replace('','unknown')
df['license'] = ['lowerlicense' if li.startswith('lli') else 'unauthorized' if li.startswith('sense') else li for li in df.license]
df['license'] = df['license'].str.replace('es desconeix','unknown')
#vehicle license_seniority
df['license_seniority'] = df['license_seniority'].str.lower().replace('desconegut','-999').astype(float)
df.loc[df['license_seniority'] < 0,'license_seniority'] = -999.0
df.fillna({'license_seniority': -999},inplace=True)
df['license_seniority'] = df['license_seniority'].astype(int)
print('NULLS: ', df.isnull().sum().sum())
if df.isnull().sum().sum() !=0:
    print(df.isnull().sum())
a = len(old_df)
b =len(df)
final_df = pd.concat([old_df,df])
assert len(final_df) == a+ b, "Check the CONCAT"
assert final_df.isnull().sum().sum() == 0,'Check NULLS'
final_df.to_csv(paths.FILES_PATH/f'types{year}.csv',index=False)

print('LENGTH: ', final_df.shape)
print(f"DONE {prename}")




url = "https://www.youtube.com/watch?v=Udt-9J8nzGE"
webbrowser.open(url,new=1)
print('DONE')

#next steps: create a datetime column to join with the weather
#Weather for 2025 done
#Organize repo

################accidents-gu##############
2025 lon_lat_check
0    7741
Name: count, dtype: int64
SOMETHING TO DELETE in 2025?: 0
Rows deleted:  0
Rows to be fixed:  0
(7741, 18)
(7741, 18)
(7741, 18)
FINAL SHAPE:  (141338, 18)
DONE accidents-gu
################causes##############
Rows dropped:  8
DONE causes
################persones##############
Criteria to group vehicles: 
The "Road Impact" Grouping (Infrastructure & Space)
This is most useful for urban planning or traffic flow analysis. It groups vehicles by their physical footprint and how they interact with the road.

Two-Wheelers & Micro-mobility (100,808): Motorcycle, moped, bicycle, quadricycles, tricycle. (The dominant group).

Light Passenger Vehicles (53,923): Car, taxi, personal motor vehicles, SUV, camper, wagon.

Mass Transit (9,073): Bus, articulated bus, minibus, tram.

Freight & Logistics (7,407): Van, all trucks, tractor-trucks.

Specialized & Other (660): Ambulance, construction machinery, unknown, others.


NULLS:

In [45]:
import os
from pathlib import Path
def read_df2(pathname):
    print(f'Working with...{pathname}')
    encodings = ['utf-8', 'latin-1']
    for enc in encodings:
        try:
            df = pd.read_csv(pathname, encoding = enc, engine='python')
            #print(f"{pathname} successfully read with {enc}")
            return df
            break
        except:
            try:
                df = pd.read_csv(pathname, encoding = enc, sep=';')
                print(f"{pathname} successfully read with {enc}")
                return df
                break
            except UnicodeDecodeError:
                continue
    else:
        raise Exception(f"unable to decode the file {pathname}")

columnes =['num_incident','vehicle_model','vehicle_brand_name',
           'vehicle_color','license','license_seniority']

total_df = pd.DataFrame()
pathname = '/Users/fcbnyc/Desktop/BarcelonAccidents/'
files = os.listdir(pathname)
for file in files:
    if file.startswith('.') is False:
        #print(file)
        file_path = pathname + file
        df = read_df2(file_path)
        df =fixing_columns(df)
        df = df[columnes]
        assert df.isnull().sum().sum() == 0, f"Check nulls from {file[:4]}"
        #print(df.shape)
        total_df =pd.concat([total_df,df])
        print(total_df.shape)
total_df.to_csv(paths.FILES_PATH/'vehicles2024.csv',index=False)

Working with.../Users/fcbnyc/Desktop/BarcelonAccidents/2010_accidents_vehicles_gu_bcn.csv
(17382, 6)
Working with.../Users/fcbnyc/Desktop/BarcelonAccidents/2024_accidents_vehicles_gu_bcn.csv
(33575, 6)
Working with.../Users/fcbnyc/Desktop/BarcelonAccidents/2017_accidents_vehicles_gu_bcn.csv
(53370, 6)
Working with.../Users/fcbnyc/Desktop/BarcelonAccidents/2018_accidents_vehicles_gu_bcn.csv
(72449, 6)
Working with.../Users/fcbnyc/Desktop/BarcelonAccidents/2023_accidents_vehicles_gu_bcn.csv
(86806, 6)
Working with.../Users/fcbnyc/Desktop/BarcelonAccidents/2025_accidents_vehicles_gu_bcn.csv
(96004, 6)
Working with.../Users/fcbnyc/Desktop/BarcelonAccidents/2011_accidents_vehicles_gu_bcn.csv
(112807, 6)
Working with.../Users/fcbnyc/Desktop/BarcelonAccidents/2022_accidents_vehicles_gu_bcn.csv
(127708, 6)
Working with.../Users/fcbnyc/Desktop/BarcelonAccidents/2019_accidents_vehicles_gu_bcn.csv
(146732, 6)
Working with.../Users/fcbnyc/Desktop/BarcelonAccidents/2016_accidents_vehicles_gu_bcn.cs